# SkeletNet-Vid: End-to-End CCTV Anomaly Detection

**Colab-ready implementation notebook**  
Pipeline: video discovery → clip sampling → person/pose extraction → Kalman smoothing → skeleton + scene descriptors → quantum-inspired feature selection → Ghost-style temporal classifier → severity scoring → chatbot-style explanation → evaluation and exports.

> **Dataset protocol:** The supplied Ano-AAD description provides video-level `normal` and `abnormal` labels (354 abnormal, 41 normal). This notebook therefore implements a reproducible **binary anomaly detector**. It does not invent fighting/theft/abuse labels that are not established by the supplied dataset description. Split is performed by source video before window generation to avoid clip-level leakage. Report the measured results produced by this notebook; the paper's 99.19% is not hard-coded or guaranteed.

**Implementation note:** To keep this executable on a standard Colab GPU, the notebook uses Ultralytics YOLO pose for person/keypoint extraction and MobileNetV3 for scene features. These are practical lightweight implementations of the described detection/pose and scene branches; they are not literal OpenPose+AlphaPose fusion or a pretrained Mobile-ConvNeXt. The optimizer is a reproducible quantum-inspired binary search (golden-sine + fractional memory + probabilistic tunnelling) followed by predator-prey local refinement; it is not a hardware-QAOA execution.

## 1. Runtime and reproducibility

In [ ]:
# Run in Google Colab. Select Runtime > Change runtime type > GPU.
import os, sys, random, json, time, math, glob, shutil, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
    print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
except Exception as e:
    print("Torch setup:", e)

SAVE_DIR = Path("/content/noel")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", SAVE_DIR)

In [ ]:
# Install the core dependencies.
%pip -q install ultralytics timm opencv-python-headless scikit-learn seaborn tqdm joblib

## 2. Dataset setup

In [ ]:
# Option A (recommended): upload/extract the Ano-AAD dataset into Google Drive,
# then set DATASET_ROOT to the folder containing normal/abnormal subfolders.
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS PATH to your extracted dataset directory.
DATASET_ROOT = Path("/content/drive/MyDrive/Ano-AAD")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg"}

# Optional automatic search for a nested dataset folder.
if not DATASET_ROOT.exists():
    candidates = [p for p in Path("/content/drive/MyDrive").rglob("*")
                  if p.is_dir() and p.name.lower() in {"ano-aad", "anomalous-action-detection-dataset"}]
    if candidates: DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())

In [ ]:
# Build a manifest from directory names. Accepted label folder aliases:
# normal/normal videos and abnormal/anomaly/anomalous/crime.
LABEL_ALIASES = {
    "normal": 0, "normal videos": 0, "normal_video": 0,
    "abnormal": 1, "anomaly": 1, "anomalous": 1, "abnormal videos": 1,
    "abnormal_video": 1, "crime": 1
}
records = []
if DATASET_ROOT.exists():
    for p in DATASET_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            label = None
            for part in p.parts:
                key = part.lower().replace("-", " ").replace("_", " ")
                if key in LABEL_ALIASES:
                    label = LABEL_ALIASES[key]
            if label is not None:
                records.append({"video_path": str(p), "label": int(label),
                                "class_name": "abnormal" if label else "normal"})
manifest = pd.DataFrame(records).drop_duplicates("video_path")
print("Videos found:", len(manifest))
if len(manifest):
    display(manifest.groupby(["class_name"]).size().rename("videos").to_frame())
    display(manifest.head())
else:
    print("No videos found. Arrange files under folders named normal/ and abnormal/,")
    print("or edit LABEL_ALIASES / DATASET_ROOT to match your dataset layout.")

In [ ]:
# If folder names are not class labels, you can manually create a CSV manifest:
# manifest = pd.read_csv('/content/drive/MyDrive/ano_aad_manifest.csv')
# Required columns: video_path,label (0=normal, 1=abnormal)
if len(manifest):
    manifest = manifest[manifest.video_path.map(lambda x: Path(x).exists())].reset_index(drop=True)
    assert set(manifest.label.unique()).issubset({0,1})
    print(manifest.label.value_counts().rename(index={0:"normal",1:"abnormal"}))

## 3. Video-level split (before clip extraction)

In [ ]:
from sklearn.model_selection import train_test_split

if len(manifest) < 4:
    raise RuntimeError("Need at least four labeled source videos. Check DATASET_ROOT / manifest.")
# Stratify at source-video level; clips from one video can never cross partitions.
train_val, test_videos = train_test_split(
    manifest, test_size=0.20, random_state=SEED, stratify=manifest.label)
train_videos, val_videos = train_test_split(
    train_val, test_size=0.20, random_state=SEED, stratify=train_val.label)
for name, df in [("train",train_videos),("validation",val_videos),("test",test_videos)]:
    print(name, len(df), df.label.value_counts().to_dict())

## 4. Frame sampling, person pose, scene and motion features

In [ ]:
import cv2
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
CLIP_LEN = 16             # frames per temporal sample
FRAME_STRIDE = 8          # sampled frames between clip starts
MAX_CLIPS_PER_VIDEO = 80  # increase for full experiments
MAX_PERSONS = 3

# COCO keypoint graph (17 joints) used by YOLO pose.
SKELETON_EDGES = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),
                  (5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]

In [ ]:
# Lightweight pretrained pose model. Downloads weights on first run.
pose_model = YOLO("yolov8n-pose.pt")
# MobileNetV3 scene backbone (ImageNet pretrained).
scene_model = timm.create_model("mobilenetv3_small_100", pretrained=True, num_classes=0,
                                global_pool="avg").to(DEVICE).eval()
import torchvision.transforms as T
from PIL import Image
scene_tf = T.Compose([T.Resize((224,224)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

In [ ]:
def kalman_smooth_pose(sequence, process_var=1e-3, measurement_var=2e-2):
    # sequence: [T,17,3] (x,y,confidence), normalized x/y.
    # Independent constant-velocity Kalman filters for each joint coordinate.
    seq = sequence.copy()
    Tn, J, _ = seq.shape
    out = np.zeros((Tn,J,2), np.float32)
    for j in range(J):
        for d in range(2):
            x = np.array([seq[0,j,d], 0.0], np.float32)
            P = np.eye(2, dtype=np.float32)
            F = np.array([[1,1],[0,1]],np.float32)
            H = np.array([[1,0]],np.float32)
            Q = process_var*np.eye(2,dtype=np.float32)
            R = np.array([[measurement_var]],np.float32)
            for t in range(Tn):
                x = F @ x; P = F @ P @ F.T + Q
                z = seq[t,j,d]
                if seq[t,j,2] > 0.05:
                    K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
                    x = x + (K[:,0] * (z - (H @ x)[0]))
                    P = (np.eye(2)-K@H)@P
                out[t,j,d] = x[0]
    return out

def skeleton_descriptor(kp):
    # kp [17,3], normalized coordinates + confidence; graph-relative and motion cues.
    xy = kp[:,:2]; conf = kp[:,2:3]
    centered = xy - xy.mean(axis=0, keepdims=True)
    scale = np.linalg.norm(centered, axis=1).mean() + 1e-6
    norm = centered / scale
    edge = np.array([np.linalg.norm(norm[a]-norm[b]) for a,b in SKELETON_EDGES],np.float32)
    return np.concatenate([norm.flatten(), conf.flatten(), edge]).astype(np.float32)

@torch.no_grad()
def scene_descriptor(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    x = scene_tf(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    return scene_model(x).squeeze(0).detach().cpu().numpy().astype(np.float32)

def read_video_frames(path, sample_fps=5, max_frames=2000):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, int(round(fps/sample_fps)))
    frames=[]; idx=0
    while len(frames)<max_frames:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            frame = cv2.resize(frame,(IMG_SIZE,IMG_SIZE))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def extract_video_clips(path, label, video_id):
    frames = read_video_frames(path)
    if len(frames) < CLIP_LEN: return []
    # YOLO pose returns person boxes and 17 COCO keypoints.
    frame_features=[]
    for frame in frames:
        result = pose_model.predict(frame, imgsz=224, conf=0.25, classes=[0],
                                    verbose=False, device=0 if DEVICE=="cuda" else "cpu")[0]
        persons=[]
        if result.keypoints is not None and result.keypoints.data is not None:
            kps = result.keypoints.data.detach().cpu().numpy()
            boxes = result.boxes.xyxy.detach().cpu().numpy() if result.boxes is not None else np.zeros((len(kps),4))
            for i,k in enumerate(kps[:MAX_PERSONS]):
                k[:,0] /= IMG_SIZE; k[:,1] /= IMG_SIZE
                persons.append((k, boxes[i] if i<len(boxes) else np.zeros(4)))
        # Fixed-size pooled skeleton descriptor (mean over detected people).
        if persons:
            sk = np.mean([skeleton_descriptor(p[0]) for p in persons],axis=0)
            centers = np.array([[(b[0]+b[2])/(2*IMG_SIZE),(b[1]+b[3])/(2*IMG_SIZE)] for _,b in persons])
            if len(centers)>1:
                dist = np.linalg.norm(centers[:,None]-centers[None,:],axis=-1)
                prox = float(dist[np.triu_indices(len(centers),1)].mean())
            else: prox = 1.0
        else:
            sk = np.zeros(17*2+17+len(SKELETON_EDGES),np.float32); prox=1.0
        scene = scene_descriptor(frame)
        frame_features.append(np.concatenate([sk,scene,[len(persons)/MAX_PERSONS,prox]]))
    arr=np.asarray(frame_features,np.float32)
    clips=[]
    for start in range(0,len(arr)-CLIP_LEN+1,FRAME_STRIDE):
        clips.append({"X":arr[start:start+CLIP_LEN],"y":int(label),"video_id":str(video_id),
                      "start_frame":start})
        if len(clips)>=MAX_CLIPS_PER_VIDEO: break
    return clips

## 5. Extract clips and cache features

In [ ]:
# Feature extraction can take time because each frame runs pose detection + scene CNN.
# Cache is stored under /content/noel so reruns do not repeat extraction.
CACHE_PATH = SAVE_DIR / "skeletnet_vid_features.npz"

def extract_partition(df, split_name):
    all_clips=[]
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Extract {split_name}"):
        all_clips.extend(extract_video_clips(row.video_path,row.label,Path(row.video_path).stem))
    print(split_name, "clips:",len(all_clips))
    return all_clips

if CACHE_PATH.exists():
    z=np.load(CACHE_PATH,allow_pickle=True)
    X_all=z["X"]; y_all=z["y"]; vid_all=z["video_id"]; split_all=z["split"]
else:
    partitions=[]
    for split_name, part in [("train",train_videos),("val",val_videos),("test",test_videos)]:
        clips=extract_partition(part,split_name)
        for c in clips: partitions.append((c["X"],c["y"],c["video_id"],split_name))
    if not partitions: raise RuntimeError("No clips extracted. Check video files/codecs.")
    X_all=np.stack([p[0] for p in partitions])
    y_all=np.array([p[1] for p in partitions],np.int64)
    vid_all=np.array([p[2] for p in partitions])
    split_all=np.array([p[3] for p in partitions])
    np.savez_compressed(CACHE_PATH,X=X_all,y=y_all,video_id=vid_all,split=split_all)
print("Feature tensor:",X_all.shape)

In [ ]:
# Split arrays retain the video-level partition.
X_train,y_train = X_all[split_all=="train"],y_all[split_all=="train"]
X_val,y_val = X_all[split_all=="val"],y_all[split_all=="val"]
X_test,y_test = X_all[split_all=="test"],y_all[split_all=="test"]
print(X_train.shape,X_val.shape,X_test.shape)
print("Feature dimension:",X_train.shape[-1])

## 6. Quantum-inspired feature selection (QDGSFO + local predator-prey refinement)

In [ ]:
# Fit feature selection on TRAIN only to prevent information leakage.
# Objective combines validation-free train discrimination and sparsity.
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

F = X_train.shape[-1]
Xtr_flat = X_train.mean(axis=1)
mi = mutual_info_classif(Xtr_flat,y_train,random_state=SEED)
mi = np.nan_to_num(mi)
mi_rank = np.argsort(mi)[::-1]

def mask_fitness(mask, X, y, alpha=0.08):
    idx=np.flatnonzero(mask)
    if len(idx)==0: return 1e6
    # Fast class-separation proxy; maximize between-class separation / within-class spread.
    Z=X[:,idx]
    mu=Z.mean(axis=0)
    between=0.0; within=0.0
    for c in np.unique(y):
        Zc=Z[y==c]
        between += len(Zc)*np.sum((Zc.mean(axis=0)-mu)**2)
        within += np.sum((Zc-Zc.mean(axis=0))**2)
    score=between/(within+1e-8)
    return -score + alpha*(len(idx)/X.shape[1])

def qdgsfo_select(X,y,pop_size=18,iters=18,keep_ratio=0.35):
    rng=np.random.default_rng(SEED); D=X.shape[1]
    # Candidate masks initialized by chaotic logistic map and MI-ranked bias.
    pop=[]
    for i in range(pop_size):
        x=rng.random(D)
        x=(x < (keep_ratio*(0.6+0.8*rng.random()))).astype(np.int8)
        x[mi_rank[:max(1,int(D*0.08))]]=1
        pop.append(x)
    pop=np.asarray(pop)
    best=pop[0].copy(); bestfit=mask_fitness(best,X,y)
    memory=np.zeros(D)
    golden=(np.sqrt(5)-1)/2
    for t in range(iters):
        frac=(1-(t+1)/(iters+1))**0.7
        for i in range(pop_size):
            phase=2*np.pi*rng.random()
            # Golden-sine global movement + fractional memory term.
            prob=1/(1+np.exp(-np.clip((pop[i]-0.5)+golden*np.sin(phase)*(best-pop[i])+frac*memory,-8,8)))
            candidate=(rng.random(D)<prob).astype(np.int8)
            # Quantum-inspired tunnelling: probabilistic bit flips.
            tunnel=rng.random(D)<(0.015+0.08*frac)
            candidate[tunnel]=1-candidate[tunnel]
            # Keep a minimum informative subset.
            if candidate.sum()<max(4,int(D*0.05)): candidate[mi_rank[:max(4,int(D*0.05))]]=1
            f=mask_fitness(candidate,X,y)
            if f < mask_fitness(pop[i],X,y): pop[i]=candidate
            if f < bestfit: best,bestfit=candidate.copy(),f
        memory=0.7*memory+0.3*(best.astype(float)-0.5)
        # Predator-prey local refinement around current best.
        for _ in range(4):
            cand=best.copy()
            flips=rng.choice(D,size=max(1,int(D*0.03)),replace=False)
            cand[flips]=1-cand[flips]
            f=mask_fitness(cand,X,y)
            if f<bestfit: best,bestfit=cand,f
    return np.flatnonzero(best),bestfit

selected_idx, selection_score = qdgsfo_select(Xtr_flat,y_train)
print("Selected features:",len(selected_idx),"of",F,"| objective:",selection_score)

## 7. SkeletNet-Lite: Ghost-style blocks + SE + temporal dilated residual blocks

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GhostModule1D(nn.Module):
    def __init__(self,in_ch,out_ch,ratio=2,kernel=1):
        super().__init__()
        init_ch=int(math.ceil(out_ch/ratio)); new_ch=out_ch-init_ch
        self.primary=nn.Sequential(nn.Conv1d(in_ch,init_ch,kernel,padding=kernel//2,bias=False),
                                   nn.BatchNorm1d(init_ch),nn.SiLU())
        self.cheap=(nn.Sequential(nn.Conv1d(init_ch,new_ch,3,padding=1,groups=init_ch,bias=False),
                                  nn.BatchNorm1d(new_ch),nn.SiLU()) if new_ch>0 else None)
        self.out_ch=out_ch
    def forward(self,x):
        y=self.primary(x)
        if self.cheap is not None: y=torch.cat([y,self.cheap(y)],dim=1)
        return y[:,:self.out_ch]

class SEBlock1D(nn.Module):
    def __init__(self,ch,ratio=4):
        super().__init__()
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Sequential(nn.Conv1d(ch,max(4,ch//ratio),1),nn.ReLU(),
                              nn.Conv1d(max(4,ch//ratio),ch,1),nn.Sigmoid())
    def forward(self,x): return x*self.fc(self.pool(x))

class TDRB(nn.Module):
    def __init__(self,ch,dilation):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv1d(ch,ch,3,padding=dilation,dilation=dilation,groups=ch,bias=False),
                                nn.BatchNorm1d(ch),nn.Conv1d(ch,ch,1,bias=False),
                                nn.BatchNorm1d(ch),nn.SiLU(),SEBlock1D(ch))
    def forward(self,x): return F.silu(x+self.conv(x))

class SkeletNetLite(nn.Module):
    def __init__(self,n_features,n_classes=2):
        super().__init__()
        self.stem=GhostModule1D(n_features,96)
        self.blocks=nn.Sequential(TDRB(96,1),TDRB(96,2),TDRB(96,4))
        self.head=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Flatten(),
                                nn.Dropout(0.25),nn.Linear(96,64),nn.SiLU(),
                                nn.Dropout(0.15),nn.Linear(64,n_classes))
    def forward(self,x): # [B,T,F]
        x=x.transpose(1,2)
        return self.head(self.blocks(self.stem(x)))

model=SkeletNetLite(len(selected_idx)).to(DEVICE)
print(model)

## 8. Focal-Tversky loss and training

In [ ]:
class FocalTverskyLoss(nn.Module):
    def __init__(self,alpha=0.4,beta=0.6,gamma=0.75,eps=1e-6):
        super().__init__(); self.alpha=alpha; self.beta=beta; self.gamma=gamma; self.eps=eps
    def forward(self,logits,target):
        probs=torch.softmax(logits,dim=1)
        onehot=F.one_hot(target,num_classes=logits.shape[1]).float()
        tp=(probs*onehot).sum(dim=0)
        fp=(probs*(1-onehot)).sum(dim=0)
        fn=((1-probs)*onehot).sum(dim=0)
        ti=(tp+self.eps)/(tp+self.alpha*fp+self.beta*fn+self.eps)
        return ((1-ti)**self.gamma).mean()

def make_loader(X,y,shuffle=False,batch_size=64):
    ds=torch.utils.data.TensorDataset(torch.tensor(X[:, :, selected_idx],dtype=torch.float32),
                                      torch.tensor(y,dtype=torch.long))
    return torch.utils.data.DataLoader(ds,batch_size=batch_size,shuffle=shuffle,
                                       num_workers=2,pin_memory=(DEVICE=="cuda"))

train_loader=make_loader(X_train,y_train,True)
val_loader=make_loader(X_val,y_val,False)
test_loader=make_loader(X_test,y_test,False)
criterion=FocalTverskyLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",patience=3,factor=0.5)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def evaluate_loader(net,loader):
    net.eval(); ys=[]; ps=[]; probs=[]
    with torch.no_grad():
        for xb,yb in loader:
            logits=net(xb.to(DEVICE))
            pr=torch.softmax(logits,1).cpu().numpy()
            probs.extend(pr); ps.extend(pr.argmax(1)); ys.extend(yb.numpy())
    return np.asarray(ys),np.asarray(ps),np.asarray(probs)

EPOCHS=30
history={"loss":[],"val_accuracy":[],"val_f1":[]}
best_f1=-1; best_path=SAVE_DIR/"skeletnet_lite_best.pt"; patience=7; stale=0
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(xb),yb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0)
        optimizer.step(); losses.append(loss.item())
    vy,vp,_=evaluate_loader(model,val_loader)
    vacc=accuracy_score(vy,vp); vf1=f1_score(vy,vp,average="macro",zero_division=0)
    history["loss"].append(float(np.mean(losses)))
    history["val_accuracy"].append(float(vacc)); history["val_f1"].append(float(vf1))
    scheduler.step(vf1)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} loss={np.mean(losses):.4f} val_acc={vacc:.4f} val_macroF1={vf1:.4f}")
    if vf1>best_f1:
        best_f1=vf1; stale=0; torch.save(model.state_dict(),best_path)
    else:
        stale+=1
        if stale>=patience: print("Early stopping"); break
model.load_state_dict(torch.load(best_path,map_location=DEVICE))
pd.DataFrame(history).to_csv(SAVE_DIR/"training_history.csv",index=False)

## 9. Test evaluation, class-wise metrics, ROC and confusion matrix

In [ ]:
from sklearn.metrics import (classification_report,confusion_matrix,roc_auc_score,
                             roc_curve,precision_score,recall_score,f1_score,
                             matthews_corrcoef,balanced_accuracy_score)
import matplotlib.pyplot as plt
import seaborn as sns

yt,yp,prob=evaluate_loader(model,test_loader)
print(classification_report(yt,yp,target_names=["normal","abnormal"],digits=4,zero_division=0))
cm=confusion_matrix(yt,yp,labels=[0,1])
tn,fp,fn,tp=cm.ravel()
specificity=tn/(tn+fp+1e-12); npv=tn/(tn+fn+1e-12)
auc=roc_auc_score(yt,prob[:,1]) if len(np.unique(yt))==2 else float("nan")
metrics={
 "Accuracy":accuracy_score(yt,yp),
 "Precision":precision_score(yt,yp,zero_division=0),
 "Sensitivity":recall_score(yt,yp,zero_division=0),
 "Specificity":specificity,
 "F1":f1_score(yt,yp,zero_division=0),
 "NPV":npv,"MCC":matthews_corrcoef(yt,yp),"ROC_AUC":auc,
 "Balanced_Accuracy":balanced_accuracy_score(yt,yp)
}
display(pd.DataFrame([metrics]).T.rename(columns={0:"Test value"}))
pd.DataFrame([metrics]).to_excel(SAVE_DIR/"test_metrics.xlsx",index=False)
pd.DataFrame(classification_report(yt,yp,labels=[0,1],target_names=["normal","abnormal"],
                                   output_dict=True,zero_division=0)).T.to_excel(
                                       SAVE_DIR/"classwise_metrics.xlsx")

In [ ]:
fig,ax=plt.subplots(figsize=(6.5,5.5))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Normal","Abnormal"],
            yticklabels=["Normal","Abnormal"],ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("SkeletNet-Vid confusion matrix")
fig.tight_layout(); fig.savefig(SAVE_DIR/"confusion_matrix.png",dpi=300,bbox_inches="tight"); plt.show()

if len(np.unique(yt))==2:
    fpr,tpr,_=roc_curve(yt,prob[:,1])
    fig,ax=plt.subplots(figsize=(6.5,5))
    ax.plot(fpr,tpr,lw=2,label=f"SkeletNet-Vid (AUC={auc:.4f})")
    ax.plot([0,1],[0,1],"--",color="gray")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("ROC curve"); ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(SAVE_DIR/"roc_curve.png",dpi=300,bbox_inches="tight"); plt.show()

## 10. Severity scoring and alert prioritization

In [ ]:
def severity_score(duration_s, mean_person_distance, crowd_density, aggression=0.5):
    # Normalize interpretable event cues to [0,1].
    d=np.clip(duration_s/30.0,0,1)
    # Smaller interpersonal distance implies higher risk.
    proximity=np.clip(1.0-mean_person_distance/0.8,0,1)
    crowd=np.clip(crowd_density/5.0,0,1)
    evidence=np.array([d,proximity,crowd,np.clip(aggression,0,1)],float)
    weights=np.array([0.25,0.30,0.20,0.25])
    # Quantum-inspired amplitude interference term, bounded to keep a valid score.
    amplitudes=np.sqrt(np.clip(evidence,0,1))
    interference=0.05*np.sin(np.pi*amplitudes[0])*np.sin(np.pi*amplitudes[1])
    score=float(np.clip(np.dot(weights,evidence)+interference,0,1))
    level="HIGH" if score>=0.70 else ("MEDIUM" if score>=0.40 else "LOW")
    return score,level

# Example values are placeholders for demonstrating the function; replace with
# measured event metadata from the tracked CCTV clip before operational use.
severity_score(duration_s=8,mean_person_distance=0.25,crowd_density=3,aggression=0.7)

## 11. Chatbot-assisted decision support (local, rule-grounded)

In [ ]:
def surveillance_assistant(predicted_class, confidence, severity, score, clip_id="current clip"):
    label="ABNORMAL ACTIVITY" if int(predicted_class)==1 else "NORMAL ACTIVITY"
    if int(predicted_class)==0:
        return f"{clip_id}: {label} (confidence {confidence:.1%}). No anomaly alert is indicated by the model."
    action={"HIGH":"Immediate operator review and verification; follow the site's emergency protocol.",
            "MEDIUM":"Prioritize review and inspect the surrounding frames for context.",
            "LOW":"Log the event and continue monitoring."}[severity]
    return (f"{clip_id}: {label} detected with model confidence {confidence:.1%}. "
            f"Severity: {severity} ({score:.2f}/1.00). {action} "
            "This is an automated screening result, not proof that a crime occurred.")
def ask_surveillance_bot(question, predicted_class=None, confidence=None, severity=None, score=None):
    q=question.lower()
    if predicted_class is None:
        return "Please run video analysis first so I can explain an actual model output."
    report=surveillance_assistant(predicted_class,confidence,severity,score)
    if any(w in q for w in ["why","reason","explain"]):
        return report+" Severity is based on duration, proximity, crowd density and an aggression cue."
    if any(w in q for w in ["what","detect","result","class"]): return report
    if any(w in q for w in ["action","do","respond"]): return "Recommended response: "+report.split("Severity:")[-1]
    return report

print(ask_surveillance_bot("Explain the result",1,0.94,"HIGH",0.82))

## 12. Single-video inference

In [ ]:
def analyze_video(video_path):
    clips=extract_video_clips(video_path,label=0,video_id=Path(video_path).stem)
    if not clips: raise ValueError("Video is too short or could not be decoded.")
    X=np.stack([c["X"] for c in clips])[:,:,selected_idx]
    loader=torch.utils.data.DataLoader(torch.tensor(X,dtype=torch.float32),batch_size=64)
    model.eval(); probs=[]
    with torch.no_grad():
        for xb in loader: probs.extend(torch.softmax(model(xb.to(DEVICE)),1)[:,1].cpu().numpy())
    p=float(np.mean(probs))
    pred=int(p>=0.5)
    # Use conservative metadata defaults; replace with tracked event measurements for deployment.
    score,level=severity_score(duration_s=len(clips)*FRAME_STRIDE/5,
                               mean_person_distance=0.5,crowd_density=1,aggression=p)
    return {"video":str(video_path),"prediction":"abnormal" if pred else "normal",
            "abnormal_probability":p,"severity_score":score,"severity":level,
            "operator_summary":surveillance_assistant(pred,p if pred else 1-p,level,score,Path(video_path).name)}

# Example:
# result = analyze_video('/content/drive/MyDrive/test_clip.mp4')
# print(json.dumps(result,indent=2))

## 13. Model size, latency and export

In [ ]:
import time
model.eval()
dummy=torch.randn(1,CLIP_LEN,len(selected_idx),device=DEVICE)
for _ in range(10):
    with torch.no_grad(): _=model(dummy)
if DEVICE=="cuda": torch.cuda.synchronize()
t0=time.perf_counter()
N=100
for _ in range(N):
    with torch.no_grad(): _=model(dummy)
if DEVICE=="cuda": torch.cuda.synchronize()
latency_ms=(time.perf_counter()-t0)*1000/N
params=sum(p.numel() for p in model.parameters())
print(f"Parameters: {params:,}")
print(f"Classifier latency: {latency_ms:.3f} ms / clip (classifier only)")
print(f"Selected features: {len(selected_idx)}/{F}")
torch.save({"model_state_dict":model.state_dict(),"selected_idx":selected_idx,
            "feature_dim":F,"clip_len":CLIP_LEN,"seed":SEED},
           SAVE_DIR/"skeletnet_vid_checkpoint.pt")
np.save(SAVE_DIR/"selected_feature_indices.npy",selected_idx)

## 14. Reproducibility report and output files

In [ ]:
report={
 "framework":"SkeletNet-Vid practical implementation",
 "dataset_root":str(DATASET_ROOT),
 "video_counts":{"train":len(train_videos),"validation":len(val_videos),"test":len(test_videos)},
 "clip_counts":{"train":len(X_train),"validation":len(X_val),"test":len(X_test)},
 "split_unit":"source video (before clip extraction)",
 "seed":SEED,"clip_length":CLIP_LEN,"frame_stride":FRAME_STRIDE,
 "feature_dimension":int(F),"selected_feature_count":int(len(selected_idx)),
 "metrics":{k:float(v) for k,v in metrics.items()},
 "parameters":int(params),"classifier_latency_ms_per_clip":float(latency_ms),
 "notes":["Binary normal/abnormal labels based on supplied dataset description.",
          "YOLOv8n-pose + MobileNetV3 implementation; not literal OpenPose/AlphaPose fusion.",
          "Quantum-inspired heuristic; no quantum hardware/QAOA execution.",
          "Severity/chatbot outputs assist human review and do not establish criminal conduct."]
}
with open(SAVE_DIR/"reproducibility_report.json","w") as f: json.dump(report,f,indent=2)
print("Saved files:")
for p in sorted(SAVE_DIR.iterdir()): print(p.name, f"({p.stat().st_size/1024:.1f} KB)")

## Important experimental reporting notes

1. Keep the source-video split fixed and report the number of source videos and generated clips in each partition.
2. If multiple segments from a single source video are generated, they must remain in the same partition.
3. The dataset description supplied here supports binary normal/abnormal evaluation. Do not report per-crime classes (fight/theft/abuse) unless you obtain verified clip-level labels for those classes.
4. Compare baselines using the same video-level split and the same clip sampling protocol. Do not copy the paper's target metrics into the results.
5. The measured classifier latency excludes frame decoding, YOLO pose inference and scene-feature extraction; report end-to-end latency separately for deployment.
6. For a stronger study, repeat the experiment across several video-level seeds and report mean ± standard deviation, plus confidence intervals.